# 14 — QLoRA, Quantization, Precision, and Memory

**Network LLM Engineering — Part III — Adaptation**

### Learning goals
- Understand FP32/FP16/BF16/INT8/4-bit
- Load a 4-bit base model for QLoRA
- Know what quantization changes and does not change

In [ ]:
%pip install -q transformers==5.14.1 datasets==5.0.1 accelerate==1.14.0 peft==0.20.0 trl==1.10.0 sentence-transformers==5.7.0 pandas matplotlib scikit-learn requests jsonschema bitsandbytes

## Quantization

Quantization stores/operates on model values with fewer bits. It reduces memory and may improve serving economics.
It is **not** free: quality, hardware support, kernel support, and numerical behavior must be tested.

**QLoRA** commonly means:
- base model weights loaded in 4-bit,
- base weights frozen,
- LoRA adapters trained in higher-precision compute.

In [ ]:
# Approximate raw weight memory ignoring metadata/overheads.
params = 8_000_000_000
for bits in [32,16,8,4]:
    gb = params * bits / 8 / 1e9
    print(f"8B params at {bits:2d}-bit ~= {gb:5.1f} GB raw weights")

In [ ]:
# GPU-only QLoRA loading pattern.
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

quant_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(quant_cfg)
# model = AutoModelForCausalLM.from_pretrained(
#     "Qwen/Qwen3-4B",
#     quantization_config=quant_cfg,
#     device_map="auto",
# )

## Terms

- **NF4:** 4-bit type designed for normally distributed weights.
- **mixed precision:** use lower precision for speed/memory while retaining selected higher-precision operations.
- **gradient checkpointing:** recompute activations to save memory.
- **FlashAttention:** optimized exact attention kernels.

### Exercise

Why can a 4-bit 8B model still require much more than 4 GB of GPU memory during training?
Think beyond stored base weights: activations, LoRA weights, gradients, optimizer state, temporary buffers, and KV/attention state.